In [25]:
import pandas as pd
import numpy as np
import re
import nltk
import os
import keras

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

In [26]:
# Download and load the data
f_path_1 = "data/train.csv"
url_1 = "https://jrssbcrsefilesnait.blob.core.windows.net/3950data1/train.csv.zip"
if not os.path.exists(f_path_1):
    file_1 = keras.utils.get_file(f_path_1, url_1)
train_df = pd.read_csv(f_path_1)

f_path_2 = "data/test.csv"
url_2 = "https://jrssbcrsefilesnait.blob.core.windows.net/3950data1/test.csv.zip"
if not os.path.exists(f_path_2):
    file_2 = keras.utils.get_file(f_path_2, url_2)
test_df = pd.read_csv(f_path_2)

## Project 1 - NLP and Text Classification

For this project you will need to classify some angry comments into their respective category of angry. The process that you'll need to follow is (roughly):
<ol>
<li> Use NLP techniques to process the training data. 
<li> Train model(s) to predict which class(es) each comment is in.
    <ul>
    <li> A comment can belong to any number of classes, including none. 
    </ul>
<li> Generate predictions for each of the comments in the test data. 
<li> Write your test data predicitions to a CSV file, which will be scored. 
</ol>

You can use any models and NLP libraries you'd like. 

## Training Data

Use the training data to train your prediction model(s). Each of the classification output columns (toxic to the end) is a human label for the comment_text, assessing if it falls into that category of "rude". A comment may fall into any number of categories, or none at all. Membership in one output category is <b>independent</b> of membership in any of the other classes (think about this when you plan on how to make these predictions - it may also make it easier to split work amongst a team...). 

In [27]:
#train_df = pd.read_csv("train.csv.zip")
train_df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\r\nWhy the edits made under my use...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\r\nMore\r\nI can't make any real suggestions...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


## Test Data

In [28]:
#test_df = pd.read_csv("test.csv")
test_df.head()

,id,comment_text
0,1,Yo bitch Ja Rule is more succesful then you'll...
1,2,== From RfC == \r\n\r\n The title is fine as i...
2,3,""" \r\n\r\n == Sources == \r\n\r\n * Zawe Ashto..."
3,4,":If you have a look back at the source, the in..."
4,5,I don't anonymously edit articles at all.


## Output Details, Submission Info, and Example Submission

For this project, please output your predictions in a CSV file. The structure of the CSV file should match the structure of the example below. 

The output should contain one row for each row of test data, complete with the columns for ID and each classification.

Into Moodle please submit:
<ul>
<li> Your notebook file(s). I'm not going to run them, just look. 
<li> Your sample submission CSV. This will be evaluated for accuracy against the real labels; only a subset of the predictions will be scored. 
</ul>

It is REALLY, REALLY, REALLY important the the structure of your output matches the specifications. The accuracies will be calculated by a script, and it is expecting a specific format. 

### Sample Evaluator

The file prediction_evaluator.ipynb contains an example scoring function, scoreChecker. This function takes a sumbission and an answer key, loops through, and evaluates the accuracy. You can use this to verify the format of your submission. I'm going to use the same function to evaluate the accuracy of your submission, against the answer key (unless I made some mistake in this counting function).

In [29]:
#Construct dummy data for a sample output. 
#You won't do this part, you have real data
#Your data should have the same structure, so the CSV output is the same
dummy_ids = ["dfasdf234", "asdfgw43r52", "asdgtawe4", "wqtr215432"]
dummy_toxic = [0,0,0,0]
dummy_severe = [0,0,0,0]
dummy_obscene = [0,1,1,0]
dummy_threat = [0,1,0,1]
dummy_insult = [0,0,1,0]
dummy_ident = [0,1,1,0]
columns = ["id", "toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
sample_out = pd.DataFrame( list(zip(dummy_ids, dummy_toxic, dummy_severe, dummy_obscene, dummy_threat, dummy_insult, dummy_ident)),
                    columns=columns)
sample_out.head()

,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,dfasdf234,0,0,0,0,0,0
1,asdfgw43r52,0,0,1,1,0,1
2,asdgtawe4,0,0,1,0,1,1
3,wqtr215432,0,0,0,1,0,0


In [30]:
#Write DF to CSV. Please keep the "out.csv" filename. Moodle will auto-preface it with an identifier when I download it. 
#This command should work with your dataframe of predictions. 
sample_out.to_csv('output/out.csv', index=False)  

## Grading

The grading for this is split between accuracy and well written code:
<ul>
<li> 75% - Accuracy. The most accurate will get 100% on this, the others will be scaled down from there. 
<li> 25% - Code quality. Can the code be followed and made sense of - i.e. comments, sections, titles. 
</ul>

## NLP Processing and Model Training

In [31]:
# Download the list of common English stopwords (e.g., "the", "is", "and")
# These are used to remove unimportant words during text preprocessing
nltk.download('stopwords')

# Download the WordNet lexical database used for lemmatization
# Lemmatization reduces words to their base form (e.g., "running" -> "run")
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

### Text Preprocessing

In [32]:
# Load English stopwords
stop_words = set(stopwords.words('english'))

# Create lemmatizer object
lemmatizer = WordNetLemmatizer()

# Function to clean text
def clean_text(text):

    text = text.lower()  # convert text to lowercase

    text = re.sub(r'[^a-zA-Z]', ' ', text)  # remove numbers and punctuation

    words = text.split()  # split text into words

    # remove stopwords and apply lemmatization
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]

    return " ".join(words)  # join words back into sentence

In [33]:
# Apply text cleaning function to training and test comments
train_df["clean_text"] = train_df["comment_text"].apply(clean_text)
test_df["clean_text"] = test_df["comment_text"].apply(clean_text)

### Feature Extraction using TF-IDF

In [34]:
# Convert text into TF-IDF numerical features
vectorizer = TfidfVectorizer(
    max_features=60000,
    ngram_range=(1,2),   # use unigrams and bigrams
    stop_words='english',
    sublinear_tf=True,
    min_df=2
)

# Fit on training data and transform it
X_train = vectorizer.fit_transform(train_df["clean_text"])

# Transform test data using the same vectorizer
X_test = vectorizer.transform(test_df["clean_text"])

### Model Training

In [35]:
# Select the target labels for multi-label classification
y_train = train_df[[
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]]

In [36]:
# Split training data into training and validation sets (80/20)
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42
)

### Train Model

In [37]:
# Initialize One-vs-Rest classifier with LinearSVC
model = OneVsRestClassifier(LinearSVC())

# Train the model on the training split
model.fit(X_train_split, y_train_split)

OneVsRestClassifier(estimator=LinearSVC())

### Calculate Accuracy

In [38]:
# Predict validation data
val_predictions = model.predict(X_val)

# Calculate accuracy
accuracy = accuracy_score(y_val, val_predictions)

print("Validation Accuracy:", accuracy)

Validation Accuracy: 0.918439605201316


In [39]:
# Define the output columns
columns = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

print("Classification Report:", classification_report(y_val, val_predictions,target_names=columns))

Classification Report:                precision    recall  f1-score   support

        toxic       0.86      0.69      0.76      3056
 severe_toxic       0.50      0.25      0.33       321
      obscene       0.89      0.69      0.78      1715
       threat       0.60      0.20      0.30        74
       insult       0.79      0.57      0.66      1614
identity_hate       0.74      0.28      0.40       294

    micro avg       0.84      0.62      0.71      7074
    macro avg       0.73      0.45      0.54      7074
 weighted avg       0.83      0.62      0.70      7074
  samples avg       0.06      0.06      0.06      7074



c:\Users\USER\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\USER\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\USER\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Prediction on Test Data

In [40]:
# Predict labels for the test data
predictions = model.predict(X_test)

# Convert predictions to a DataFrame
pred_df = pd.DataFrame(predictions, columns=columns)

# Add the 'id' column from the test set
pred_df.insert(0, "id", test_df["id"])

In [41]:
# Save predictions to CSV for submission
pred_df.to_csv("output/out.csv", index=False)

# Display first few rows of predictions
pred_df.head()

,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,1,1,0,1,0,1,0
1,2,0,0,0,0,0,0
2,3,0,0,0,0,0,0
3,4,0,0,0,0,0,0
4,5,0,0,0,0,0,0


### Model & Approach Description

- **Model:** LinearSVC with One-vs-Rest strategy for multi-label classification.  
- **Text Processing:** Lowercased text, removed non-letter characters, removed stopwords, and applied lemmatization.  
- **Features:** TF-IDF vectorization with unigrams and bigrams.  
- **Validation:** Training data split 80/20 for validation.  
- **Validation Accuracy:** 0.918 (~91.8%).  
- **weighted F1-score of 0.70**, indicating good overall performance.  
- Performance is higher for common classes such as *toxic* and *obscene*, while rarer classes like *threat* have lower recall due to limited training samples.
- **Output:** Predictions for test data saved to `output/out.csv`.